In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.io import loadmat

In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection"
)

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "CWRU"
)

PROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "CWRU"
)

WINDOW_PATH = (
    PROCESSED_PATH
    / "windows"
)

WINDOW_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw:", RAW_PATH)
print("Windows:", WINDOW_PATH)

Raw: C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\raw\CWRU
Windows: C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows


In [3]:
FS = 12000

WINDOW_SECONDS = 1

WINDOW_SIZE = (
    FS * WINDOW_SECONDS
)

OVERLAP = 0.5

STEP_SIZE = int(
    WINDOW_SIZE * (1 - OVERLAP)
)

print("Sampling frequency:", FS)
print("Window size:", WINDOW_SIZE)
print("Overlap:", OVERLAP)
print("Step size:", STEP_SIZE)

Sampling frequency: 12000
Window size: 12000
Overlap: 0.5
Step size: 6000


In [4]:
CLASS_MAP = {
    "NORMAL": "Healthy",
    "B007": "Ball",
    "IR007": "Inner Race",
    "OR007@6": "Outer Race"
}

In [6]:
def get_class(filename):

    if filename.startswith("NORMAL"):
        return "Healthy"

    elif filename.startswith("B007"):
        return "Ball"

    elif filename.startswith("IR007"):
        return "Inner Race"

    elif filename.startswith("OR007@6"):
        return "Outer Race"

    return "Unknown"
print(get_class("NORMAL_0.mat"))
print(get_class("B007_0.mat"))
print(get_class("IR007_0.mat"))
print(get_class("OR007@6_0.mat"))

Healthy
Ball
Inner Race
Outer Race


In [7]:
file_path = (
    RAW_PATH
    / "B007_0.mat"
)

data = loadmat(
    file_path
)

de_keys = [
    key for key in data.keys()
    if key.endswith("_DE_time")
]

print(de_keys)

['X118_DE_time']


In [8]:
signal = np.asarray(
    data[de_keys[0]]
).squeeze()

print(
    "Total samples:",
    len(signal)
)

Total samples: 122571


In [9]:
num_windows = (
    len(signal) - WINDOW_SIZE
) // STEP_SIZE + 1

print(
    "Number of windows:",
    num_windows
)

Number of windows: 19


In [10]:
all_windows = []

mat_files = sorted(
    RAW_PATH.glob("*.mat")
)

print(
    "MAT files found:",
    len(mat_files)
)

MAT files found: 16


In [11]:
for file_path in mat_files:

    data = loadmat(file_path)

    de_keys = [
        key for key in data.keys()
        if key.endswith("_DE_time")
    ]

    if not de_keys:
        print(
            "Skipping - no DE signal:",
            file_path.name
        )
        continue

    class_name = get_class(
        file_path.name
    )

    for de_key in de_keys:

        signal = np.asarray(
            data[de_key]
        ).squeeze()

        recording_id = (
            file_path.stem
            + "_"
            + de_key.replace(
                "_DE_time",
                ""
            )
        )

        num_windows = (
            len(signal) - WINDOW_SIZE
        ) // STEP_SIZE + 1

        print(
            file_path.name,
            de_key,
            "Class:",
            class_name,
            "Windows:",
            num_windows
        )

        for window_id in range(
            num_windows
        ):

            start = (
                window_id *
                STEP_SIZE
            )

            end = (
                start +
                WINDOW_SIZE
            )

            window = signal[
                start:end
            ]

            all_windows.append({
                "recording_id": recording_id,
                "source_file": file_path.name,
                "signal_id": de_key.replace(
                    "_DE_time",
                    ""
                ),
                "class": class_name,
                "window_id": window_id,
                "start_sample": start,
                "end_sample": end
            })

B007_0.mat X118_DE_time Class: Ball Windows: 19
B007_1.mat X119_DE_time Class: Ball Windows: 19
B007_2.mat X120_DE_time Class: Ball Windows: 19
B007_3.mat X121_DE_time Class: Ball Windows: 19
IR007_0.mat X105_DE_time Class: Inner Race Windows: 19
IR007_1.mat X106_DE_time Class: Inner Race Windows: 19
IR007_2.mat X107_DE_time Class: Inner Race Windows: 19
IR007_3.mat X108_DE_time Class: Inner Race Windows: 19
NORMAL_0.mat X097_DE_time Class: Healthy Windows: 39
NORMAL_1.mat X098_DE_time Class: Healthy Windows: 79
NORMAL_2.mat X098_DE_time Class: Healthy Windows: 79
NORMAL_2.mat X099_DE_time Class: Healthy Windows: 79
NORMAL_3.mat X100_DE_time Class: Healthy Windows: 79
OR007@6_0.mat X130_DE_time Class: Outer Race Windows: 19
OR007@6_1.mat X131_DE_time Class: Outer Race Windows: 19
OR007@6_2.mat X132_DE_time Class: Outer Race Windows: 19
OR007@6_3.mat X133_DE_time Class: Outer Race Windows: 19


In [12]:
window_df = pd.DataFrame(
    all_windows
)

print(
    "Total windows:",
    len(window_df)
)

display(
    window_df.head()
)

Total windows: 583


,recording_id,source_file,signal_id,class,window_id,start_sample,end_sample
0,B007_0_X118,B007_0.mat,X118,Ball,0,0,12000
1,B007_0_X118,B007_0.mat,X118,Ball,1,6000,18000
2,B007_0_X118,B007_0.mat,X118,Ball,2,12000,24000
3,B007_0_X118,B007_0.mat,X118,Ball,3,18000,30000
4,B007_0_X118,B007_0.mat,X118,Ball,4,24000,36000


In [13]:
print(
    window_df["class"].value_counts()
)

class
Healthy       355
Ball           76
Inner Race     76
Outer Race     76
Name: count, dtype: int64


In [14]:
windows_per_recording = (
    window_df
    .groupby(
        [
            "recording_id",
            "class"
        ]
    )
    .size()
    .reset_index(
        name="window_count"
    )
)

display(
    windows_per_recording
)

,recording_id,class,window_count
0,B007_0_X118,Ball,19
1,B007_1_X119,Ball,19
2,B007_2_X120,Ball,19
3,B007_3_X121,Ball,19
4,IR007_0_X105,Inner Race,19
5,IR007_1_X106,Inner Race,19
6,IR007_2_X107,Inner Race,19
7,IR007_3_X108,Inner Race,19
8,NORMAL_0_X097,Healthy,39
9,NORMAL_1_X098,Healthy,79


In [16]:
print(
    "Unknown classes:",
    (
        window_df["class"]
        == "Unknown"
    ).sum()
)
print(
    window_df.isnull().sum()
)

Unknown classes: 0
recording_id    0
source_file     0
signal_id       0
class           0
window_id       0
start_sample    0
end_sample      0
dtype: int64


In [17]:
window_metadata_path = (
    WINDOW_PATH
    / "cwru_window_metadata.csv"
)

window_df.to_csv(
    window_metadata_path,
    index=False
)

print(
    "Saved:",
    window_metadata_path
)

Saved: C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\cwru_window_metadata.csv


In [18]:
recordings = (
    window_df[
        [
            "recording_id",
            "class"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Unique recordings:",
    len(recordings)
)

display(recordings)

Unique recordings: 17


,recording_id,class
0,B007_0_X118,Ball
1,B007_1_X119,Ball
2,B007_2_X120,Ball
3,B007_3_X121,Ball
4,IR007_0_X105,Inner Race
5,IR007_1_X106,Inner Race
6,IR007_2_X107,Inner Race
7,IR007_3_X108,Inner Race
8,NORMAL_0_X097,Healthy
9,NORMAL_1_X098,Healthy


In [19]:
print(
    recordings["class"]
    .value_counts()
)

class
Healthy       5
Ball          4
Inner Race    4
Outer Race    4
Name: count, dtype: int64


In [20]:
recordings.to_csv(
    WINDOW_PATH
    / "recording_groups.csv",
    index=False
)